
# Module 2a — Preparing the LLM model in Azure AI Foundry

> Part of the **"Develop & Deploy AI Agents on Azure with LangChain, Python and Foundry"** course.

## 🎯 Learning objectives

By the end of this module you will be able to:

1. Create an **Azure AI Foundry project** with Terraform.
2. Deploy a **chat model** (GPT-class) into the project.
3. Retrieve the **endpoint** and **API key** of the deployed model.
4. Call the model from Python using the **OpenAI-compatible API** that Foundry exposes.
5. Understand the difference between a *raw LLM call* and an *agent call* — which we will build on top in the next modules.

## 🧠 Key concepts

| Concept                | What it means                                                                              |
| ---------------------- | ------------------------------------------------------------------------------------------ |
| **Azure AI Foundry**   | Managed Azure service that hosts foundation models behind a secure, billable endpoint.     |
| **Project**            | A workspace inside Foundry that groups model deployments, datasets, evaluations, agents.   |
| **Model deployment**   | A specific model version (e.g. `gpt-4o-mini`) exposed under a stable URL with a quota.     |
| **OpenAI-compatible**  | Foundry exposes `/openai/v1/...` so any OpenAI SDK / LangChain `ChatOpenAI` just works.    |

## 📋 Prerequisites

- Azure subscription with permission to deploy AI resources.
- Terraform installed and logged in (`az login`).
- The `infra/` folder of this course (already provisioned in earlier course step).

## 🗺️ Where this fits

```
┌────────────────────────────────────────────┐
│  Module 2a (you are here)                  │
│  → LLM lives in Foundry (managed)          │
├────────────────────────────────────────────┤
│  Module 2b                                 │
│  → LLM lives in Container Apps + GPU       │
│    (self-hosted Gemma)                     │
└────────────────────────────────────────────┘
```

Both produce an **OpenAI-compatible endpoint**, so the agent code in later modules will not need to change — only the `base_url` and `api_key`.

---


# Testing access to Foundry LLM via Python SDK

This notebook demonstrates how to connect to an Azure Foundry LLM deployment using the OpenAI Python SDK. 

Make sure you have the `openai` package installed. You can install it using pip:

In [ ]:
%pip install azure-ai-projects azure-identity openai

Get the required values for Foundry endpoint, deployment name, and API key. You can get these values from the Azure portal or using Terraform outputs as shown below:

In [6]:
foundry_endpoint = ! terraform output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n + "openai/v1/"
print ("endpoint:" + foundry_endpoint)

llm_model_deployment_name = ! terraform output -raw llm_model_deployment_name
llm_model_deployment_name = llm_model_deployment_name.n
print ("llm_model_deployment_name:" + llm_model_deployment_name)

foundry_api_key =  ! terraform output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print ("foundry_api_key:" + foundry_api_key)

endpoint:https://foundry-555.cognitiveservices.azure.com/openai/v1/
llm_model_deployment_name:gpt-5.2
foundry_api_key:7lDCTqHKRZDATNxBKqZk3QKh8EoFlRVvMIYGeJds0fWad81JYiLAJQQJ99CDACfhMk5XJ3w3AAAAACOGlf3w


Then you can use the following code to create a client and make a request to the deployed model:

In [5]:
from openai import OpenAI

client = OpenAI(
    base_url=endpoint,
    api_key=foundry_api_key
)

response = client.responses.create(
    model=llm_model_deployment_name,
    input="What is the capital of France?",
)

print(f"answer: {response.output[0]}")

answer: ResponseOutputMessage(id='msg_08662a056f6ed0e00069e4841ee13c819791ec049a83ff4f9d', content=[ResponseOutputText(annotations=[], text='Paris.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)


In [ ]:
import os
from dotenv import load_dotenv

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, WebSearchPreviewTool, ApproximateLocation

load_dotenv()

project_client = AIProjectClient(
    endpoint=foundry_endpoint,
    credential=DefaultAzureCredential(),
)

openai_client = project_client.get_openai_client()

from azure.ai.projects.models import PromptAgentDefinition, WebSearchPreviewTool, ApproximateLocation

agent = project_client.agents.create_version(
    agent_name="MyAgent",
    definition=PromptAgentDefinition(
        model=llm_model_deployment_name, # os.environ["FOUNDRY_MODEL_DEPLOYMENT_NAME"],
        instructions="You are a helpful assistant that can search the web",
        tools=[
            WebSearchPreviewTool()
        ],
    ),
    description="Agent for web search.",
)

ResourceNotFoundError: (404) Resource not found
Code: 404
Message: Resource not found